In [78]:
import os
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
import numpy as onp

from qiskit import QuantumCircuit
from qiskit_ionq import IonQProvider

from pennylane.templates import StronglyEntanglingLayers

In [79]:
provider = IonQProvider(token = "MkU7QmnnoPUCcmylhNo9pPgiGqrEJL2E")

/Users/dev/anaconda3/lib/python3.11/site-packages/qiskit_ionq/helpers.py:659: UserWarning: Failed to get qubits for simulator: HTTPSConnectionPool(host='api.ionq.co', port=443): Read timed out. (read timeout=5). Using 4
  warnings.warn(


In [80]:
noisy_backend = provider.get_backend("ionq_simulator")
noisy_backend.set_options(noise_model="aria-1")

In [81]:
degree = 2
coeffs = [ 0.15 + 0.15j]*degree
coeff0 = 0.1

def target_function(x):
    func = coeff0
    for idx,coeff in enumerate(coeffs):
        exponent = complex(0, (idx + 1) * x) 
        func += coeff * np.exp(exponent) + np.conjugate(coeff) * np.exp(-exponent)
        
    return np.real(func)

In [82]:
n_qubits = degree
n_ansatz_layers = 2

In [83]:
def S(x):
    for w in range(n_qubits):
        qml.RY(x, wires = w)

def W(theta):
    # theta shape: (n_ansatz_layers, n_qubits, 3)
    for l in range(n_ansatz_layers):
        # single-qubit trainable rotations
        for w in range(n_qubits):
            qml.Rot(theta[l, w, 0], theta[l, w, 1], theta[l, w, 2], wires=w)

        # simple entangler
        for w in range(n_qubits - 1):
            qml.CNOT(wires=[w, w + 1])

In [84]:
def square_loss(y, predictions):
    return np.mean((y - predictions) ** 2)

def cost(params, x, y):
    predictions = circuit_parallel(params, x)   # x can now be a whole batch
    return square_loss(y, predictions)


In [89]:
dev = qml.device('qiskit.remote', wires = n_qubits, backend = noisy_backend)
@qml.set_shots(2000)
@qml.batch_input(argnum = n_ansatz_layers * n_qubits * 3)
@qml.qnode(dev, diff_method = "parameter-shift")

def circuit_parallel(params, x):
    W(params[0])
    S(x)
    W(params[1])
    return qml.expval(qml.PauliZ(0))
    
    



In [90]:
X = onp.array(np.linspace(-6, 6, 70))

target_y = onp.array([target_function(x) for x in X])

In [91]:
params_parallel = np.array(2 * np.pi * onp.random.random( size = (2, n_ansatz_layers, n_qubits, 3)), requires_grad=True)
print(params_parallel)

[[[[1.30375169 2.86430317 4.15034958]
   [5.95637833 2.58919191 0.50976838]]

  [[3.75845201 1.95476035 0.10271813]
   [3.35685485 3.99463945 2.39510392]]]


 [[[2.12687522 6.05334867 3.48772895]
   [1.34551156 5.18547023 1.52879476]]

  [[0.56180097 0.80438395 1.00659145]
   [5.41369411 2.83182824 2.13108195]]]]


In [92]:
cst = [cost(params_parallel, X, target_y)]

opt = qml.AdamOptimizer(0.3)
n_iter = 100
batch_size = 20

cst = [cost(params_parallel, X, target_y)]

opt = qml.AdamOptimizer(0.3)
n_iter = 100
batch_size = 20

for steps in range(n_iter):
    batch_index = np.random.randint(0, len(X), (batch_size,))
    x_batch = X[batch_index]
    y_batch = target_y[batch_index]

    params_parallel = opt.step(lambda p: cost(p, x_batch, y_batch), params_parallel)

    c = cost(params_parallel, X, target_y)
    cst.append(c)

    if (steps + 1) % 10 == 0:
        print( "Cost at step {0:3}: {1}".format(steps+1, c))
    

IBMInputValueError: 'The instruction rzz is supported only for angles in the range [0, pi/2], but an angle (-1.45124580977145) outside of this range has been requested.'